# RAPIDEXPLAIN — Interactive Credit Card Fraud Detector
### Veritas AI Hackathon | Enter transaction features → FRAUD / NOT FRAUD

This version does **not** require uploading a sample CSV to make a prediction. It gives you an input box directly in the notebook. Paste one transaction's feature values and click **DETECT TRANSACTION**.

The original LightGBM model from your first notebook is embedded here. This notebook does not train on the two demo CSVs.

**Important:** the trained model expects the anonymized columns `Time`, `V1`–`V28`, and `Amount`. These are engineered/PCA-style dataset features, not ordinary card details. It cannot classify a transaction from only its dollar amount, merchant name, card number, or location. Do not enter real card numbers, CVVs, PINs, or payment credentials.


## 1. Install and load the existing model

This loads the exact saved model from your first analysis. It does not retrain it.

In [ ]:
%pip -q install lightgbm==4.6.0 scikit-learn==1.6.1 joblib==1.5.2 ipywidgets


In [ ]:
import base64, io, csv, warnings, joblib, numpy as np, pandas as pd
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

warnings.filterwarnings("ignore")

MODEL_B64 = '''...'''
bundle = joblib.load(io.BytesIO(base64.b64decode(MODEL_B64)))

model = bundle["model"]
scaler = bundle["scaler"]
FEATURES = bundle["features"]

# High-recall threshold from the first notebook's validation threshold analysis.
# Lower thresholds catch more possible fraud but also create more false alarms.
THRESHOLD_HIGH_RECALL = float(bundle["threshold_cost"])
THRESHOLD_BALANCED = float(bundle["threshold_f1"])

INPUT_COLUMNS = ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount"]

print("Loaded model:", bundle["model_name"])
print("Input format: Time, V1, V2, ... V28, Amount")
print(f"High-recall threshold: {THRESHOLD_HIGH_RECALL:.4f}")
print(f"Balanced threshold:    {THRESHOLD_BALANCED:.4f}")


Loaded model: LightGBM
Input format: Time, V1, V2, ... V28, Amount
High-recall threshold: 0.1353
Balanced threshold:    0.3476


## 2. Enter transaction data here

Paste a **single CSV row** containing 30 numeric values in this order:

`Time, V1, V2, ... V28, Amount`

You can paste either:
- One row of 30 comma-separated values, or
- A two-line CSV containing the header and one transaction row.

The `FRAUD.csv` and `NON_FRAUD.csv` demo files supplied with this project use this exact format. They are synthetic examples selected to demonstrate the frozen model's two output classes; they are not training data or proof that all real fraud will be caught.

In [ ]:

from google.colab import files
import pandas as pd
import io
import numpy as np
from IPython.display import display, HTML

display(HTML("""
<h2>RAPIDEXPLAIN</h2>
<h3>Upload Transaction File</h3>
<p>Select a CSV file to analyze for credit card fraud.</p>
"""))

uploaded = files.upload()

if not uploaded:
    raise ValueError("No file uploaded.")

required = ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount"]

all_results = []

for filename, content in uploaded.items():

    df = pd.read_csv(io.BytesIO(content))

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{filename}: Missing columns: {missing}")

    for col in required:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    if df[required].isna().any().any():
        raise ValueError(f"{filename}: Missing or invalid feature values.")

    if (df["Amount"] < 0).any():
        raise ValueError(f"{filename}: Negative transaction amount.")

    prepared = df.copy()

    prepared["Amount_log"] = np.log1p(prepared["Amount"])

    t = np.mod(prepared["Time"].to_numpy(), 86400)

    prepared["Hour_sin"] = np.sin(2 * np.pi * t / 86400)
    prepared["Hour_cos"] = np.cos(2 * np.pi * t / 86400)

    X = prepared[FEATURES]
    X_scaled = scaler.transform(X)

    probabilities = model.predict_proba(X_scaled)[:, 1]

    threshold = THRESHOLD_HIGH_RECALL

    predictions = np.where(
        probabilities >= threshold,
        "FRAUD",
        "NOT FRAUD"
    )

    results = df.copy()
    results["FRAUD PROBABILITY"] = probabilities
    results["FINAL RESULT"] = predictions

    all_results.append(results)

    for i, result in results.iterrows():

        label = result["FINAL RESULT"]
        probability = result["FRAUD PROBABILITY"]

        color = "#dc2626" if label == "FRAUD" else "#16a34a"

        display(HTML(f"""
        <div style="border:3px solid {color};
                    padding:20px;border-radius:12px;
                    margin:12px 0;text-align:center">
            <h2 style="color:{color}">{label}</h2>
            <h3>Fraud Probability: {probability:.2%}</h3>
        </div>
        """))

final_results = pd.concat(all_results, ignore_index=True)

final_results.to_csv("Fraud_Detection_Results.csv", index=False)

print("Analysis complete!")
files.download("Fraud_Detection_Results.csv")

Textarea(value='', description='INPUT DATA:', layout=Layout(height='150px', width='100%'), placeholder='Time,V…

Dropdown(description='Detection mode:', layout=Layout(width='100%'), options=(('High recall — catch more possi…

Button(button_style='danger', description='DETECT TRANSACTION', icon='search', layout=Layout(height='42px', wi…

Output()

## 3. Run the detector

The notebook validates the pasted values, recreates the same time/amount features used by the original model, and returns a binary result. RapidExplain calculates feature contributions only for transactions flagged as FRAUD.

In [ ]:

from google.colab import files
import pandas as pd
import io
import numpy as np
from IPython.display import display, HTML

display(HTML("""
<h2>RAPIDEXPLAIN</h2>
<h3>Upload Transaction File</h3>
<p>Select a CSV file to analyze for credit card fraud.</p>
"""))

uploaded = files.upload()

if not uploaded:
    raise ValueError("No file uploaded.")

required = ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount"]

all_results = []

for filename, content in uploaded.items():

    df = pd.read_csv(io.BytesIO(content))

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{filename}: Missing columns: {missing}")

    for col in required:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    if df[required].isna().any().any():
        raise ValueError(f"{filename}: Missing or invalid feature values.")

    if (df["Amount"] < 0).any():
        raise ValueError(f"{filename}: Negative transaction amount.")

    prepared = df.copy()

    prepared["Amount_log"] = np.log1p(prepared["Amount"])

    t = np.mod(prepared["Time"].to_numpy(), 86400)

    prepared["Hour_sin"] = np.sin(2 * np.pi * t / 86400)
    prepared["Hour_cos"] = np.cos(2 * np.pi * t / 86400)

    X = prepared[FEATURES]
    X_scaled = scaler.transform(X)

    probabilities = model.predict_proba(X_scaled)[:, 1]

    threshold = THRESHOLD_HIGH_RECALL

    predictions = np.where(
        probabilities >= threshold,
        "FRAUD",
        "NOT FRAUD"
    )

    results = df.copy()
    results["FRAUD PROBABILITY"] = probabilities
    results["FINAL RESULT"] = predictions

    all_results.append(results)

    for i, result in results.iterrows():

        label = result["FINAL RESULT"]
        probability = result["FRAUD PROBABILITY"]

        color = "#dc2626" if label == "FRAUD" else "#16a34a"

        display(HTML(f"""
        <div style="border:3px solid {color};
                    padding:20px;border-radius:12px;
                    margin:12px 0;text-align:center">
            <h2 style="color:{color}">{label}</h2>
            <h3>Fraud Probability: {probability:.2%}</h3>
        </div>
        """))

final_results = pd.concat(all_results, ignore_index=True)

final_results.to_csv("Fraud_Detection_Results.csv", index=False)

print("Analysis complete!")
files.download("Fraud_Detection_Results.csv")


## 4. Optional: try the two separate demo files

Two separate CSVs are supplied with this notebook:
- `FRAUD.csv` — a synthetic feature row that this frozen model scores above the high-recall threshold.
- `NON_FRAUD.csv` — a synthetic feature row that this frozen model scores below the threshold.

Open either CSV, copy its one data row (not the header), paste it into the input box above, and click **DETECT TRANSACTION**. These files are test fixtures only; they are not used to train or modify the model.

The demo rows were generated synthetically and selected by checking the existing model's output. They are useful for verifying that both UI outcomes work, but they are not real, independently verified fraud/legitimate transactions.

## 5. What the model can and cannot do

- **Can:** score inputs represented in the same `Time`, `V1`–`V28`, `Amount` feature format used during training.
- **Cannot:** accept arbitrary raw transaction details (for example, merchant, city, card number, or just amount) and magically convert them into the training features.
- **Cannot guarantee:** that every fraudulent transaction will be detected. The first notebook's reported test recall was 78.95% at its balanced threshold. The high-recall mode uses a lower threshold selected on validation data, which tends to increase alerts and false positives; it does not guarantee a particular recall on new data.

### Existing reported test performance
The first notebook reported LightGBM PR-AUC **0.8197**, ROC-AUC **0.9716**, precision **0.9259**, recall **0.7895**, and F1 **0.8523** on its held-out test split. Those are prior experiment results, not newly measured results from this interactive notebook.

**Competition innovation — RapidExplain:** the detector gives a fast binary decision first, then calculates detailed feature contributions only for flagged transactions. This focuses explanation work on alerts and avoids computing explanations for every normal prediction.